In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Zeitspanne auswählen
dates = pd.date_range("2025-11-05", "2025-11-12")

# ✅ Hier manuell deine Task-Verläufe eintragen
remaining_tasks = [5, 5, 5, 4, 2, 2, 1,1]

# Anzahl Gesamt-Tasks aus erstem Wert ableiten
tasks_total = remaining_tasks[0]

# DataFrame
data = pd.DataFrame({
    "Date": dates,
    "Remaining Tasks": remaining_tasks
})

plt.figure(figsize=(8, 5))
plt.plot(data["Date"], data["Remaining Tasks"], marker="o", linewidth=2, label="Tatsächlicher Verlauf")

plt.title("Burndown Chart (Nov 12 – Nov 19, 2025)")
plt.xlabel("Datum")
plt.ylabel("Offene Tasks")

# ✅ Y-Achse mit extra Höhe
plt.yticks(range(0, tasks_total + 2))
plt.ylim(0, tasks_total + 1)

plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.6)

# ✅ Ideale Burndown-Linie (automatisch berechnet)
ideal_line = [tasks_total - (tasks_total / (len(dates)-1)) * i for i in range(len(dates))]
plt.plot(data["Date"], ideal_line, linestyle="--", linewidth=1.8, label="Ideale Linie")

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

# --- Datengrundlage ---
assignments = [
    ("Amine", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Andree", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Felix", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Moritz", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Sofian", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Yassine", "Weekly Meeting", "2025-11-12 16:30", "2025-11-12 17:15"),
    ("Felix", "Präsentation vorbereiten", "2025-11-18 18:15", "2025-11-18 19:00"),
    ("Moritz", "Präsentation vorbereiten", "2025-11-18 18:15", "2025-11-18 19:00"),
    ]

df = pd.DataFrame(assignments, columns=["employee", "task", "start", "end"])
df["start"] = pd.to_datetime(df["start"])
df["end"] = pd.to_datetime(df["end"])

# --- Mitarbeiter chronologisch sortieren ---
first_tasks = df.groupby("employee")["start"].min().sort_values()
employees = first_tasks.index.tolist()
y_pos = {emp: i for i, emp in enumerate(employees)}

# --- Farben pro Task ---
colors = list(mcolors.TABLEAU_COLORS.values())
task_colors = {task: colors[i % len(colors)] for i, task in enumerate(df["task"].unique())}

# --- Plot ---
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(14, 6))

# --- Hintergrund einfärben: ganze Tage ---
start_date = df["start"].dt.floor("D").min()
end_date = df["end"].dt.ceil("D").max()

curr_date = start_date
while curr_date < end_date:
    next_day = curr_date + timedelta(days=1)
    day_start = datetime.combine(curr_date, datetime.min.time())
    day_end = datetime.combine(next_day, datetime.min.time())

    if curr_date.weekday() >= 5:
        color = "#ffeaea"  # Wochenende
    else:
        color = "#dcdcdc" if curr_date.day % 2 == 0 else "#eaeaea"

    ax.axvspan(mdates.date2num(day_start), mdates.date2num(day_end),
               facecolor=color, alpha=0.9, zorder=0)
    curr_date = next_day

# --- Vertikale Linien jede 6 Stunden ---
hour_lines = pd.date_range(start=start_date, end=end_date, freq="6H")
for h in hour_lines:
    ax.axvline(mdates.date2num(h), color="lightgray", linestyle="--", linewidth=0.6, zorder=1)

# --- Gantt-Balken ---
for _, r in df.iterrows():
    start_num = mdates.date2num(r["start"])
    end_num = mdates.date2num(r["end"])
    width = end_num - start_num
    y = y_pos[r["employee"]]

    ax.barh(y, width, left=start_num, height=0.45,
            color=task_colors[r["task"]], edgecolor="black", linewidth=0.8, zorder=2)

# --- Achsen & Formatierung ---
ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(list(y_pos.keys()), fontsize=10)
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m\n%a'))
ax.set_xlabel("Datum / Uhrzeit", fontsize=10)

ax.set_xlim([
    mdates.date2num(datetime.combine(start_date, datetime.min.time())),
    mdates.date2num(datetime.combine(end_date, datetime.min.time()))
])

ax.set_title(f"Gantt-Diagramm: {start_date:%d.%m.%Y} – {end_date:%d.%m.%Y}",
             fontsize=12, weight="bold")

# --- Legende ---
patches = [mpatches.Patch(color=c, label=t) for t, c in task_colors.items()]
legend = ax.legend(handles=patches, title="Tasks", loc='upper left',
                   bbox_to_anchor=(1.02, 1), frameon=True)

# ==========================
#  ARBEITSSTUNDEN-BERECHNUNG
# ==========================

df["hours"] = (df["end"] - df["start"]).dt.total_seconds() / 3600

# Gemeinsame Stunden werden allen Mitarbeitenden angerechnet
individual_hours = df.groupby("employee")["hours"].sum().reset_index()
individual_hours.columns = ["Mitarbeiter", "Stunden"]
individual_hours["Stunden"] = individual_hours["Stunden"].round(2)
gesamtstunden = individual_hours["Stunden"].sum().round(2)

# --- Tabelle dynamisch unter der Legende ---
fig.canvas.draw()  # notwendig, damit die Legendenposition berechnet wird
legend_box = legend.get_window_extent()
inv = fig.transFigure.inverted()
legend_box_fig = inv.transform(legend_box)
legend_x0, legend_y0, legend_x1, legend_y1 = legend_box_fig.flatten()

table_y = legend_y0 - 0.02  # kleiner Abstand unterhalb der Legende

table = plt.table(
    cellText=individual_hours.values,
    colLabels=individual_hours.columns,
    cellLoc='center',
    colLoc='center',
    loc='upper left',
    bbox=[1.02, table_y - 0.35, 0.20, 0.35]  # dynamische Position
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.1, 1.3)

# --- Gesamtstunden unter der Tabelle ---
projekt_stunden_bisher = 77.5  # ⬅️ HIER deinen bisherigen Projektstand eintragen
gesamtstunden_projekt = projekt_stunden_bisher + gesamtstunden

# Textanzeige
plt.text(
    1.02, table_y - 0.40,
    f"Gesamtstunden (Woche): {gesamtstunden:.1f} h",
    transform=ax.transAxes,
    fontsize=11,
    fontweight='bold',
)

plt.text(
    1.02, table_y - 0.47,  # etwas tiefer platziert
    f"Gesamtstunden (Projekt): {gesamtstunden_projekt:.1f} h",
    transform=ax.transAxes,
    fontsize=11,
    fontweight='bold',
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

# ==========================
# Beispiel-Daten mit Abhängigkeiten
# ==========================
planned_issues = [
    ("Feature X", 32, "Group1", 3, []),
    ("Feature Y", 24, "Group1", 4, ["Feature X"]),
    ("Bugfix A", 16, "Group2", 3, []),
    ("Feature Z", 40, "Group3", 5, ["Bugfix A"]),
    ("Feature W", 48, "Group3", 4, ["Feature Z"]),
]

df = pd.DataFrame(planned_issues, columns=["issue", "estimated_hours", "dependency_group", "planned_days", "dependencies"])

# ==========================
# Funktion: Arbeitstage addieren
# ==========================
def add_workdays(start, days):
    current = start
    added = 0
    while added < days:
        current += timedelta(days=1)
        if current.weekday() < 5:
            added += 1
    return current

# ==========================
# Start- und Enddaten berechnen
# ==========================
start_date = datetime(2025, 11, 19)
planned_starts = {}
planned_ends = {}

for _, row in df.iterrows():
    if row['dependencies']:
        dep_ends = [planned_ends[dep] for dep in row['dependencies']]
        start = add_workdays(max(dep_ends), 1)
    else:
        start = start_date
    end = add_workdays(start, row['planned_days'])
    planned_starts[row['issue']] = start
    planned_ends[row['issue']] = end

df['planned_start'] = df['issue'].map(planned_starts)
df['planned_end'] = df['issue'].map(planned_ends)

# ==========================
# Issues nach Startdatum sortieren (frühester oben)
# ==========================
df_sorted = df.sort_values("planned_start", ascending=True)
y_pos = {issue: i for i, issue in enumerate(df_sorted["issue"])}

# Farben für Dependency Groups
colors = list(mcolors.TABLEAU_COLORS.values())
groups = df_sorted["dependency_group"].unique()
group_colors = {grp: colors[i % len(colors)] for i, grp in enumerate(groups)}

# ==========================
# Plot erstellen
# ==========================
fig, ax = plt.subplots(figsize=(14, 6))

# Hintergrund (Wochenenden)
curr_date = df_sorted['planned_start'].min()
end_date = max(df_sorted['planned_end'].max(), datetime(2025, 12, 24))
while curr_date <= end_date:
    next_day = curr_date + timedelta(days=1)
    color = "#ffeaea" if curr_date.weekday() >= 5 else ("#eaeaea" if curr_date.day % 2 == 0 else "#dcdcdc")
    ax.axvspan(mdates.date2num(curr_date), mdates.date2num(next_day), facecolor=color, alpha=0.5, zorder=0)
    curr_date = next_day

# Gantt-Balken plotten
for _, r in df_sorted.iterrows():
    start_num = mdates.date2num(r["planned_start"])
    end_num = mdates.date2num(r["planned_end"] + timedelta(days=1))
    width = end_num - start_num
    y = y_pos[r["issue"]]
    ax.barh(y, width, left=start_num, height=0.9,
            color=group_colors[r["dependency_group"]],
            edgecolor="black", zorder=2)
    ax.text(start_num + width/2, y, f"{r['estimated_hours']}h",
            va='center', ha='center', fontsize=9, color='black')

# ==========================
# Meilensteine hinzufügen (Mittwoch bis Mittwoch)
# ==========================
milestone_start = datetime(2025, 11, 19)
milestone_end = datetime(2025, 12, 24)
milestone_dates = []

current = milestone_start
ms_number = 8
while current <= milestone_end:
    next_wed = current + timedelta(days=7)
    milestone_dates.append((current, next_wed, f"Milestone {ms_number}"))
    current = next_wed
    ms_number += 1

ms_y = -1
for start, end, label in milestone_dates:
    start_num = mdates.date2num(start)
    end_num = mdates.date2num(end)
    ax.barh(ms_y, end_num - start_num, left=start_num, height=0.5,
            color="#a6cee3", edgecolor="black", alpha=0.7, zorder=1)
    ax.text(start_num + (end_num-start_num)/2, ms_y, label,
            va='center', ha='center', fontsize=9, fontweight='bold', color='black')

# ==========================
# Achsen formatieren (nur Mittwoch und Sonntag beschriften)
# ==========================
ax.set_yticks(list(y_pos.values()) + [ms_y])
ax.set_yticklabels(list(y_pos.keys()) + ["Milestones"])
ax.invert_yaxis()  # Frühester Start oben

# X-Achse: nur Mittwoch (2) und Sonntag (6)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=(2, 6)))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))

ax.set_xlabel("Datum")
ax.set_title("Projektplan", fontsize=12, weight="bold")

# Legende
patches = [plt.Rectangle((0,0),1,1,color=group_colors[grp]) for grp in groups]
ax.legend(patches, groups, title="Dependency Groups", loc='upper left', bbox_to_anchor=(1.02, 1))

# Minimaler vertikaler Abstand
ax.set_ylim(ms_y-0.5, len(y_pos)-0.5)

plt.tight_layout()
plt.show()
